In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings,ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [3]:
loader = PyMuPDFLoader("../data/Data_science.pdf")
docs = loader.load()

In [4]:
print("Total documents:", len(docs))
print(docs[0].page_content[:500])
print(docs[0].metadata)

Total documents: 257
STATISTICS ........................................................................................................................................................... 6 
Q1. 
WHAT IS THE CENTRAL LIMIT THEOREM AND WHY IS IT IMPORTANT? ........................................................................ 6 
Q2. 
WHAT IS SAMPLING? HOW MANY SAMPLING METHODS DO YOU KNOW? ................................................................... 7 
Q3. 
WHAT IS THE DIFFERENCE BETWEEN TYPE I VS TYPE II ERR
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20220222215734', 'source': '../data/Data_science.pdf', 'file_path': '../data/Data_science.pdf', 'total_pages': 257, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20220222215734', 'page': 0}


In [5]:
### Splitting dat into multiple chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)

In [6]:
len(splitted_data)

667

In [7]:
## embedding generation
embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [8]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding = embeddings
)

In [9]:
query = "Machine Learning and Data Science Content"
data = vector_store.similarity_search(query=query)

In [10]:
data

[Document(id='cc338186-c6a9-4869-8bad-6b0f6fcf48e6', metadata={'subject': '', 'page': 188, 'keywords': '', 'source': '../data/Data_science.pdf', 'author': '', 'format': 'PDF 1.7', 'producer': 'PDFium', 'creationDate': 'D:20220222215734', 'moddate': '', 'total_pages': 257, 'title': '', 'modDate': '', 'trapped': '', 'creator': 'PDFium', 'file_path': '../data/Data_science.pdf', 'creationdate': 'D:20220222215734'}, page_content='gradually learn how to cluster them into groups by computing the mean of the distance between\n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \ndifferent points. \n \nQ43 Mention the difference between Data Mining and Machine learning? \nMachine learning relates to the study, design, and development of the algorithms that give\n \n \n \n \n \n \n \n \n \n \n \n \n \n \ncomputers the capability to learn without being explicitly programmed. While data mining can\n \n \n \n \n \n \n \n \n \n \n \n \n \nbe defined as the process in which the unstructured data tries to e

In [11]:
data[0]

Document(id='cc338186-c6a9-4869-8bad-6b0f6fcf48e6', metadata={'subject': '', 'page': 188, 'keywords': '', 'source': '../data/Data_science.pdf', 'author': '', 'format': 'PDF 1.7', 'producer': 'PDFium', 'creationDate': 'D:20220222215734', 'moddate': '', 'total_pages': 257, 'title': '', 'modDate': '', 'trapped': '', 'creator': 'PDFium', 'file_path': '../data/Data_science.pdf', 'creationdate': 'D:20220222215734'}, page_content='gradually learn how to cluster them into groups by computing the mean of the distance between\n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \ndifferent points. \n \nQ43 Mention the difference between Data Mining and Machine learning? \nMachine learning relates to the study, design, and development of the algorithms that give\n \n \n \n \n \n \n \n \n \n \n \n \n \n \ncomputers the capability to learn without being explicitly programmed. While data mining can\n \n \n \n \n \n \n \n \n \n \n \n \n \nbe defined as the process in which the unstructured data tries to ex

In [12]:
## Context Generation
context = ""
for doc in data:
    context += doc.page_content + "\n"
print(context)

gradually learn how to cluster them into groups by computing the mean of the distance between
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
different points. 
 
Q43 Mention the difference between Data Mining and Machine learning? 
Machine learning relates to the study, design, and development of the algorithms that give
 
 
 
 
 
 
 
 
 
 
 
 
 
 
computers the capability to learn without being explicitly programmed. While data mining can
 
 
 
 
 
 
 
 
 
 
 
 
 
be defined as the process in which the unstructured data tries to extract knowledge or unknown
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
interesting patterns.  During this processing machine, learning algorithms are used. 
Steve Nouri  ​                           ​https://www.linkedin.com/in/stevenouri/
Data Science 
 
Q1. 
What is Data Science? List the differences between supervised and 
unsupervised learning. 
 
Data Science is a blend of various tools, algorithms, and machine learning principles with the goal to 
discover hidden patterns from th

In [13]:
llm = ChatOllama(model="llama3.2:1b")

In [14]:
res = llm.invoke(f"""can you provide me the answer based on 
                 provided context for my question,context:{context} and question : {query}""")

In [15]:
print(res.content)

I'll provide a clear and concise answer to your question based on the provided context.

**Machine Learning vs. Data Science:**

Machine Learning (ML) is a subset of Data Science that involves the development of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. It is used to make predictions or decisions based on historical data.

Data Science, on the other hand, is a broader field that encompasses various tools, techniques, and methodologies for extracting insights and knowledge from data. It involves the analysis of data, identification of patterns, and creation of models to make predictions or decisions.

**Differences between Supervised and Unsupervised Learning:**

Based on the provided context, here are the differences between supervised and unsupervised learning:

- **Supervised Learning:**
  - Input data is labelled
  - Split in training/validation/test
  - Used for prediction
  - Example: classification and regressi

In [20]:
### chain-->context+llm+query
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [22]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context},
    Question: {question}
""")

In [21]:
rag_chain = get_context | prompt | llm

In [25]:
res = rag_chain.invoke("What is statistical modelling ")

In [26]:
print(res.content)

Statistical modeling is the process of analyzing and interpreting data using statistical techniques to identify patterns, relationships, and trends. It involves using mathematical and computational methods to analyze data from various sources, such as surveys, observations, and experiments, to make predictions, estimates, and inferences about a particular phenomenon or process.

Statistical modeling typically involves the following steps:

1. Data collection and preparation: Gathering and cleaning data from various sources.
2. Data analysis: Using statistical techniques, such as regression, hypothesis testing, and time series analysis, to analyze the data and identify relationships.
3. Model selection: Choosing the most appropriate statistical model based on the type of data, the research question, and the expected outcomes.
4. Model estimation and validation: Estimating the model parameters and validating the model using statistical methods.
5. Model interpretation: Interpreting the r